# Qa refraction bottlenecks

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# QA: Refraction and Path Bottlenecks

This notebook visualizes a highly refracted offshore-to-nearshore route and overlays local channel-width circles (derived from distance-to-coast) to highlight bottlenecks.

In [ ]:
from __future__ import annotations

import pickle
from pathlib import Path

import contextily as ctx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Circle
from scipy import ndimage

plt.rcParams["figure.dpi"] = 120
TARGET_EPSG = 32633


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        if all((parent / name).exists() for name in ("configs", "data", "notebooks", "src")):
            return parent
    raise FileNotFoundError("Could not resolve project root with configs/data/notebooks/src.")


PROJECT_ROOT = resolve_project_root()
ROUTING_PKL = PROJECT_ROOT / "data/processed/routing_features.pkl"
BATHY_NPZ = PROJECT_ROOT / "data/processed/bathy/bathy_field_project_site.npz"

with ROUTING_PKL.open("rb") as handle:
    routing_payload = pickle.load(handle)

routes = routing_payload.get("routes")
if not isinstance(routes, pd.DataFrame):
    raise TypeError("routing_features.pkl does not contain a DataFrame under key routes")

required_cols = [
    "site_name",
    "reachable",
    "path_rc",
    "path_xy",
    "path_bottleneck_m",
    "static_tortuosity_sum",
    "static_net_deflection_deg",
    "static_final_approach_deg",
]
missing = [col for col in required_cols if col not in routes.columns]
if missing:
    raise KeyError(f"Missing required routing columns: {missing}")

routes = routes[routes["reachable"] == True].copy()
if routes.empty:
    raise ValueError("No reachable paths found in routing_features.pkl")

selected = routes.sort_values("static_tortuosity_sum", ascending=False).iloc[6]
path_xy = np.asarray(selected["path_xy"], dtype=float)
path_rc = np.asarray(selected["path_rc"], dtype=int)
if path_xy.ndim != 2 or path_xy.shape[0] < 2:
    raise ValueError("Selected route path is not valid for plotting")

with np.load(BATHY_NPZ, allow_pickle=True) as npz:
    x_coord = np.asarray(npz["x"], dtype=float)
    y_coord = np.asarray(npz["y"], dtype=float)
    land_mask = np.asarray(npz["land_mask"], dtype=bool)

dx = float(np.median(np.diff(x_coord))) if x_coord.size > 1 else 1.0
dy = float(np.median(np.diff(y_coord))) if y_coord.size > 1 else 1.0
distance_to_coast_m = ndimage.distance_transform_edt(~land_mask, sampling=(dy, dx))
channel_width_m = 2.0 * distance_to_coast_m[path_rc[:, 0], path_rc[:, 1]]

print(f"Selected site: {selected['site_name']}")
print(f"Path points: {len(path_xy)}")
print(f"Path bottleneck (m): {float(selected['path_bottleneck_m']):.2f}")
print(
    f"Tortuosity / Net Deflection / Final Approach (deg): "
    f"{float(selected['static_tortuosity_sum']):.2f} / "
    f"{float(selected['static_net_deflection_deg']):.2f} / "
    f"{float(selected['static_final_approach_deg']):.2f}"
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

pad = 1_500.0
ax.set_xlim(float(np.min(path_xy[:, 0]) - pad), float(np.max(path_xy[:, 0]) + pad))
ax.set_ylim(float(np.min(path_xy[:, 1]) - pad), float(np.max(path_xy[:, 1]) + pad))

try:
    ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, crs="EPSG:32633", zoom=14)
except Exception as exc:
    print(f"Basemap fetch failed ({exc}); plotting without imagery.")

ax.plot(
    path_xy[:, 0],
    path_xy[:, 1],
    color="#10e0ff",
    linewidth=2.4,
    zorder=3,
    label="Routed path",
)
ax.scatter(
    path_xy[0, 0],
    path_xy[0, 1],
    c="#ffcc00",
    s=70,
    edgecolor="black",
    zorder=4,
    label="Offshore start",
)
ax.scatter(
    path_xy[-1, 0],
    path_xy[-1, 1],
    c="#ff4d4d",
    s=70,
    edgecolor="black",
    zorder=4,
    label="Nearshore end",
)

stride = max(1, len(path_xy) // 20)
sample_idx = np.arange(0, len(path_xy), stride, dtype=int)
if sample_idx[-1] != len(path_xy) - 1:
    sample_idx = np.append(sample_idx, len(path_xy) - 1)

for idx in sample_idx:
    radius = float(max(channel_width_m[idx], 1.0))
    circle = Circle(
        (float(path_xy[idx, 0]), float(path_xy[idx, 1])),
        radius=radius,
        edgecolor="#ff9d00",
        facecolor="none",
        linewidth=1.2,
        alpha=0.35,
        zorder=2,
    )
    ax.add_patch(circle)

stats_text = "\n".join(
    [
        f"Site: {selected['site_name']}",
        f"Path Bottleneck: {float(selected['path_bottleneck_m']):.2f} m",
        f"Tortuosity Sum: {float(selected['static_tortuosity_sum']):.2f} deg",
        f"Net Deflection: {float(selected['static_net_deflection_deg']):.2f} deg",
        f"Final Approach: {float(selected['static_final_approach_deg']):.2f} deg",
    ]
)

ax.text(
    0.02,
    0.98,
    stats_text,
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=10,
    color="white",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="black", alpha=0.65, edgecolor="white"),
    zorder=5,
)

ax.set_title("Refracted Route With Local Channel-Width Circles")
ax.set_xlabel("X (m), EPSG:32633")
ax.set_ylabel("Y (m), EPSG:32633")
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()